Ноутбук считает BM baseline на русском корпусе. Это лексический метод, с которым сравнивались dense retrieval модели.


In [ ]:
!pip -q install rank-bm25

import os
import re
import json
import math
import hashlib
from collections import Counter, defaultdict

import numpy as np
from rank_bm25 import BM25Okapi


ENC_DIR = "/kaggle/input/datasets/sukiss/corpus-embedings"  # где лежат main_corpus.jsonl/test_corpus.jsonl
METRIC_TESTSET = "/kaggle/input/datasets/sukiss/dataset-for-metrics/chunks_testset_metric_instructions.jsonl"

CORPUS_MODE = "combined"

OUT_DIR = "/kaggle/working/bm25_eval"
os.makedirs(OUT_DIR, exist_ok=True)

PER_QUERY_OUT = os.path.join(OUT_DIR, f"bm25_{CORPUS_MODE}_per_query.jsonl")
SUMMARY_OUT = os.path.join(OUT_DIR, f"bm25_{CORPUS_MODE}_summary.json")


def sha1_text(t):
    return hashlib.sha1(t.strip().encode("utf-8")).hexdigest()

def tokenize(text):
    return re.findall(r"[а-яёa-z0-9]+", text.lower())

def load_corpus_jsonl(path, prefix):
    docids, texts, hashes = [], [], []

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            obj = json.loads(line)
            docid = f"{prefix}::{obj['docid']}"
            text = obj["text"]

            docids.append(docid)
            texts.append(text)
            hashes.append(sha1_text(text))

    return docids, texts, hashes

def load_corpus(enc_dir, mode):
    all_docids, all_texts, all_hashes = [], [], []

    if mode in ["main", "combined"]:
        d, t, h = load_corpus_jsonl(os.path.join(enc_dir, "main_corpus.jsonl"), "main")
        all_docids += d
        all_texts += t
        all_hashes += h

    if mode in ["test", "combined"]:
        d, t, h = load_corpus_jsonl(os.path.join(enc_dir, "test_corpus.jsonl"), "test")
        all_docids += d
        all_texts += t
        all_hashes += h

    return all_docids, all_texts, all_hashes

def load_metric_items(path):
    items = []

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            it = json.loads(line)

            if all(it.get(k) for k in [
                "query_id",
                "query",
                "only_instruction",
                "reverse_instruction",
                "pmrr_instruction",
                "positive",
                "pmrr_changed_docs",
            ]):
                items.append(it)

    return items

def query_text(it, mode):
    q = it["query"].strip()

    if mode == "orig":
        return q
    if mode == "inst":
        return (q + " " + it["only_instruction"].strip()).strip()
    if mode == "rev":
        return (q + " " + it["reverse_instruction"].strip()).strip()
    if mode == "pmrr":
        return (q + " " + it["pmrr_instruction"].strip()).strip()

    raise ValueError(mode)

def mrr_at_k(rank, k):
    return 1.0 / rank if rank <= k else 0.0

def ndcg_at_k(rank, k):
    return 1.0 / math.log2(rank + 1) if rank <= k else 0.0

def ap_at_k(rank, k):
    return 1.0 / rank if rank <= k else 0.0

def hit_at_k(rank, k):
    return 1.0 if rank <= k else 0.0

def sicr_score(r_ori, s_ori, r_ins, s_ins, r_rev, s_rev):
    return float(
        (r_ins < r_ori) and
        (s_ins > s_ori) and
        (r_ori < r_rev) and
        (s_ori > s_rev)
    )

def wise_score(r_ori, r_ins, r_rev, n_positive_original=1, k=20):
    if r_ins <= r_ori < r_rev:
        if r_ori <= n_positive_original and r_ins == 1:
            return 1.0
        if r_ori <= k:
            return (1.0 - (math.sqrt(max(0, r_ori - r_ins)) / k)) * (1.0 / math.sqrt(r_ins))
        return 0.01

    if r_rev < r_ori < r_ins:
        return -1.0
    if r_ori <= r_ins:
        return (r_ori - r_ins) / r_ins
    if r_rev <= r_ori:
        return (r_rev - r_ori) / r_ori

    return 0.0

def pmrr_doc_score(rank_old, rank_new):
    rr_old = 1.0 / rank_old
    rr_new = 1.0 / rank_new

    if rank_old > rank_new:
        return (rr_old / rr_new) - 1.0
    return 1.0 - (rr_new / rr_old)

def rank_score_for_candidates(scores, candidate_indices):
    candidate_indices = list(candidate_indices)
    best_idx = max(candidate_indices, key=lambda i: scores[i])
    best_score = float(scores[best_idx])
    rank = 1 + int(np.sum(scores > best_score))
    return rank, best_score

def summarize_mode(rows, mode):
    ranks = [r[f"rank_{mode}"] for r in rows]

    return {
        "MRR@10": float(np.mean([mrr_at_k(r, 10) for r in ranks])),
        "nDCG@5": float(np.mean([ndcg_at_k(r, 5) for r in ranks])),
        "nDCG@10": float(np.mean([ndcg_at_k(r, 10) for r in ranks])),
        "MAP@1000": float(np.mean([ap_at_k(r, 1000) for r in ranks])),
        "Hit@10": float(np.mean([hit_at_k(r, 10) for r in ranks])),
    }

def summarize_rows(rows):
    pmrr_values = [r["pmrr"] for r in rows if r["pmrr"] is not None]

    return {
        "count": len(rows),
        "orig": summarize_mode(rows, "orig"),
        "inst": summarize_mode(rows, "inst"),
        "rev": summarize_mode(rows, "rev"),
        "instruction_metrics": {
            "SICR": float(np.mean([r["sicr"] for r in rows])),
            "SICR_x100": float(100 * np.mean([r["sicr"] for r in rows])),
            "WISE": float(np.mean([r["wise"] for r in rows])),
            "WISE_x100": float(100 * np.mean([r["wise"] for r in rows])),
            "pMRR": float(np.mean(pmrr_values)) if pmrr_values else None,
            "pMRR_x100": float(100 * np.mean(pmrr_values)) if pmrr_values else None,
        }
    }


docids, corpus_texts, corpus_hashes = load_corpus(ENC_DIR, CORPUS_MODE)

hash_to_indices = defaultdict(list)
for i, h in enumerate(corpus_hashes):
    hash_to_indices[h].append(i)

print("Corpus mode:", CORPUS_MODE)
print("Docs:", len(docids))

tokenized_corpus = [tokenize(t) for t in corpus_texts]
bm25 = BM25Okapi(tokenized_corpus)

print("BM25 ready")


items = load_metric_items(METRIC_TESTSET)

kept = []
missing_positive = 0

for it in items:
    pos_hash = sha1_text(it["positive"])
    if pos_hash not in hash_to_indices:
        missing_positive += 1
        continue
    kept.append(it)

items = kept

print("Metric items kept:", len(items))
print("Missing positive:", missing_positive)

rows = []
stats = Counter()

for idx, it in enumerate(items, 1):
    pos_hash = sha1_text(it["positive"])
    pos_candidates = hash_to_indices[pos_hash]

    scores_orig = bm25.get_scores(tokenize(query_text(it, "orig")))
    scores_inst = bm25.get_scores(tokenize(query_text(it, "inst")))
    scores_rev = bm25.get_scores(tokenize(query_text(it, "rev")))
    scores_pmrr = bm25.get_scores(tokenize(query_text(it, "pmrr")))

    r_ori, s_ori = rank_score_for_candidates(scores_orig, pos_candidates)
    r_ins, s_ins = rank_score_for_candidates(scores_inst, pos_candidates)
    r_rev, s_rev = rank_score_for_candidates(scores_rev, pos_candidates)

    changed_scores = []
    changed_details = []

    for ch in it.get("pmrr_changed_docs", []):
        text_hash = ch.get("text_hash") or sha1_text(ch.get("text", ""))
        cand = hash_to_indices.get(text_hash)

        if not cand:
            stats["missing_pmrr_changed_doc"] += 1
            continue

        r_old, s_old = rank_score_for_candidates(scores_inst, cand)
        r_new, s_new = rank_score_for_candidates(scores_pmrr, cand)

        score = pmrr_doc_score(r_old, r_new)
        changed_scores.append(score)

        changed_details.append({
            "doc_id": ch.get("doc_id"),
            "rank_old_inst": r_old,
            "score_old_inst": s_old,
            "rank_new_pmrr": r_new,
            "score_new_pmrr": s_new,
            "pmrr_doc_score": score,
            "reason": ch.get("reason", ""),
            "text_hash": text_hash,
        })

    row = {
        "query_id": str(it["query_id"]),
        "instruction_style": it.get("instruction_style", "unknown"),

        "rank_orig": r_ori,
        "score_orig": s_ori,
        "rank_inst": r_ins,
        "score_inst": s_ins,
        "rank_rev": r_rev,
        "score_rev": s_rev,

        "sicr": sicr_score(r_ori, s_ori, r_ins, s_ins, r_rev, s_rev),
        "wise": wise_score(r_ori, r_ins, r_rev),
        "pmrr": float(np.mean(changed_scores)) if changed_scores else None,
        "pmrr_changed_docs_eval": changed_details,
    }

    rows.append(row)

    if idx % 100 == 0:
        print(f"{idx}/{len(items)}")

with open(PER_QUERY_OUT, "w", encoding="utf-8") as f:
    for r in rows:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

summary = summarize_rows(rows)

by_style = {}
for style in sorted(set(r["instruction_style"] for r in rows)):
    style_rows = [r for r in rows if r["instruction_style"] == style]
    by_style[style] = summarize_rows(style_rows)

summary["model"] = "BM25"
summary["corpus_mode"] = CORPUS_MODE
summary["docs"] = len(docids)
summary["by_style"] = by_style
summary["stats"] = dict(stats)
summary["outputs"] = {
    "per_query": PER_QUERY_OUT,
    "summary": SUMMARY_OUT,
}

with open(SUMMARY_OUT, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("\nDONE")
print("Per-query:", PER_QUERY_OUT)
print("Summary:", SUMMARY_OUT)

print(json.dumps({
    "model": summary["model"],
    "count": summary["count"],
    "orig": summary["orig"],
    "inst": summary["inst"],
    "rev": summary["rev"],
    "instruction_metrics": summary["instruction_metrics"],
    "stats": summary["stats"],
}, ensure_ascii=False, indent=2))

print("\nBY STYLE:")
for style, s in summary["by_style"].items():
    print(style, json.dumps({
        "count": s["count"],
        "SICR_x100": s["instruction_metrics"]["SICR_x100"],
        "WISE_x100": s["instruction_metrics"]["WISE_x100"],
        "pMRR_x100": s["instruction_metrics"]["pMRR_x100"],
    }, ensure_ascii=False))
